# Assignment 3: Transformer is All You Need

**Task:** Train a small decoder-only Transformer for next-token prediction on Tiny Shakespeare, then visualize its attention and explain its learning behavior.

This notebook implements:

0. Setup and data loading (source, download date, checksum)
1. Data preparation (positional split, subword tokenizer, token windows)


## 0. Setup

Import the libraries used below and fix the constants that make the work reproducible. Hardware and package versions are recorded because training time and exact losses depend on them; every configuration is trained in the same runtime so their times are comparable.


In [1]:
import hashlib
import os
import platform
import random
import subprocess
import sys
from datetime import date
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib
import numpy as np
import torch

try:
    import tokenizers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers"])
    import tokenizers

assert hasattr(torch.nn, "RMSNorm"), "PyTorch >= 2.4 is required for nn.RMSNorm"

SEED = 42
TRAIN_FRACTION = 0.8
ARTIFACTS_DIR = Path("artifacts")  # tokenizer and selected checkpoint
ASSETS_DIR = Path("assets")  # figures used in the report
ARTIFACTS_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cpu_model():
    if platform.system() == "Linux":
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    if platform.system() == "Darwin":
        result = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
        return result.stdout.strip()
    return platform.processor() or "unknown"


set_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Python      {platform.python_version()}")
print(f"PyTorch     {torch.__version__}")
print(f"tokenizers  {tokenizers.__version__}")
print(f"NumPy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"Platform    {platform.platform()}")
print(f"CPU         {cpu_model()} ({os.cpu_count()} cores)")
if DEVICE.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU         {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB)")
print(f"Device      {DEVICE}")
print(f"Seed        {SEED}")


Python      3.9.6
PyTorch     2.8.0
tokenizers  0.22.2
NumPy       2.0.2
matplotlib  3.9.4
Platform    macOS-26.6.2-arm64-arm-64bit
CPU         Apple M5 (10 cores)
Device      mps
Seed        42


### 0.1 Load the data

Prefer a local `input.txt` in the notebook working directory (Colab upload). If it is missing, download it from the source, Andrej Karpathy's char-rnn repository, and save the download date next to it so reruns report the original date. An uploaded file is the repository copy fetched on 2026-11-21. The SHA-256 check guarantees that every run and every experiment uses exactly the same text.


In [2]:
DATA_FILE = Path("input.txt")
DATE_FILE = Path("input.txt.date")  # download date of DATA_FILE
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
EXPECTED_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
REPO_COPY_DATE = "2026-11-21"  # date the repository copy (uploaded file) was fetched from DATA_URL

if not DATA_FILE.exists():
    urlretrieve(DATA_URL, DATA_FILE)
    DATE_FILE.write_text(date.today().isoformat())
download_date = DATE_FILE.read_text().strip() if DATE_FILE.exists() else REPO_COPY_DATE

raw_bytes = DATA_FILE.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
assert sha256 == EXPECTED_SHA256, f"Unexpected file contents (SHA-256 {sha256})"
text = raw_bytes.decode("utf-8")
chars = sorted(set(text))

print(f"Source         {DATA_URL}")
print(f"Download date  {download_date}")
print(f"SHA-256        {sha256}")
print(f"Characters     {len(text):,}")
print(f"Lines          {text.count(chr(10)):,}")
print(f"Unique chars   {len(chars)}: {repr(''.join(chars))}")
print(f"Spaces         {text.count(' '):,} ({text.count(' ') / len(text):.1%} of characters)")
print()
print(text[:250])


Source         https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Download date  2026-11-21
SHA-256        86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed
Characters     1,115,394
Lines          40,000
Unique chars   65: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
Spaces         169,892 (15.2% of characters)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



**Interpretation.**

- The file matches the expected checksum: 1,115,394 characters in 40,000 lines, so every experiment below uses identical text.
- Only 65 distinct characters occur: the 52 upper- and lower-case letters, space, newline, the digit `3`, and ten punctuation marks. Spaces alone are about 15% of the text.
- The corpus is play dialogue: a speaker name on its own line ending in a colon, followed by the speech. Newlines, colons, and capitalized names are therefore frequent structural patterns that the tokenizer and model must represent.


## 1. Data preparation

The raw text is split by position before the tokenizer is fitted or any window is created, so nothing derived from validation text can influence training.


### 1.1 Positional split

The first 80% of characters is used for training and the final 20% for validation. A positional cut keeps validation text contiguous and later in the file, so the model is scored on passages it never saw rather than on shuffled lines from the same scenes.


In [3]:
split_at = int(TRAIN_FRACTION * len(text))
train_text = text[:split_at]
val_text = text[split_at:]

print(f"Split at character  {split_at:,}")
print(f"Train characters    {len(train_text):,}")
print(f"Val characters      {len(val_text):,}")
print(f"Validation characters missing from training: {sorted(set(val_text) - set(train_text))}")
print()
print(f"Before the cut: {text[split_at - 60:split_at]!r}")
print(f"After the cut:  {text[split_at:split_at + 60]!r}")


Split at character  892,315
Train characters    892,315
Val characters      223,079
Validation characters missing from training: []

Before the cut: 'ur frames,--let me be bold;\nI do arrest your words. Be that '
After the cut:  "you are,\nThat is, a woman; if you be more, you're none;\nIf y"


**Interpretation.**

- Training text is 892,315 characters and validation text is 223,079 characters.
- The cut falls mid-sentence ("Be that | you are"). This is harmless: the two halves are tokenized and windowed separately, so no window contains text from both sides.
- Every validation character also occurs in training text, so the tokenizer below never needs its unknown token on this data.


### 1.2 Subword tokenizer

A character-level BPE tokenizer (Hugging Face `tokenizers`) is fitted on `train_text` only:

- **Base alphabet:** the 65 characters of the training text. BPE then repeatedly merges the most frequent adjacent pair until the vocabulary reaches 500 entries, including the single special token `[UNK]`.
- **Pre-tokenization:** each newline is isolated as its own token, and spaces become the visible marker `▁` attached to the following word (as in SentencePiece). Merges therefore never span a space or a line break (punctuation can still join a word, as in `GLOUCESTER:`), and decoding restores the exact text.
- **Unseen symbols:** a character absent from the training text maps to `[UNK]`. It still occupies one position, so the sequence length is unchanged, but its identity is lost.
- **Why not byte-level BPE:** a byte-level tokenizer never needs `[UNK]`, but its 256 base bytes would use half of the 500-entry budget; the character alphabet leaves 434 entries for merges and so produces shorter sequences.

The fitted tokenizer is saved to `artifacts/` and reused unchanged for validation and for every configuration.


In [4]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

VOCAB_SIZE = 500  # includes the special token
UNK_TOKEN = "[UNK]"

tokenizer = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split("\n", behavior="isolated"),
    pre_tokenizers.Metaspace(prepend_scheme="never"),
])
tokenizer.decoder = decoders.Metaspace(prepend_scheme="never")
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[UNK_TOKEN], show_progress=False)
tokenizer.train_from_iterator([train_text], trainer=trainer)
tokenizer.save(str(ARTIFACTS_DIR / "tokenizer.json"))

vocab_size = tokenizer.get_vocab_size()
unk_id = tokenizer.token_to_id(UNK_TOKEN)
assert vocab_size <= VOCAB_SIZE

train_encoding = tokenizer.encode(train_text)
val_encoding = tokenizer.encode(val_text)
assert tokenizer.decode(train_encoding.ids) == train_text
assert tokenizer.decode(val_encoding.ids) == val_text

print(f"Vocabulary size     {vocab_size} (1 special + {len(set(train_text))} characters + {vocab_size - 1 - len(set(train_text))} merges)")
for name, chars, ids in [("Train", train_text, train_encoding.ids), ("Val", val_text, val_encoding.ids)]:
    print(f"{name:<5} tokens        {len(ids):,} ({len(chars) / len(ids):.2f} characters per token, {ids.count(unk_id)} [UNK])")
print()
longest = sorted(tokenizer.get_vocab(), key=lambda token: (-len(token), token))[:10]
print(f"Longest tokens:     {longest}")
print(f"Example:            {tokenizer.encode('First Citizen:' + chr(10) + 'Before we proceed').tokens}")
print(f"Unseen characters:  {tokenizer.encode('café ☃').tokens}")


Vocabulary size     500 (1 special + 65 characters + 434 merges)
Train tokens        407,182 (2.19 characters per token, 0 [UNK])
Val   tokens        105,963 (2.11 characters per token, 0 [UNK])

Longest tokens:     ['GLOUCESTER:', '▁RICHARD', 'GLOUCES', '▁father', '▁should', 'ICHARD', '▁again', '▁blood', '▁death', '▁heart']
Example:            ['First', '▁C', 'it', 'i', 'z', 'en', ':', '\n', 'B', 'e', 'fore', '▁we', '▁pro', 'ce', 'ed']
Unseen characters:  ['c', 'a', 'f', '[UNK]', '▁', '[UNK]']


**Interpretation.**

- The vocabulary is exactly 500 entries: `[UNK]`, the 65 training characters, and 434 learned merges.
- Encoding and decoding reproduce both splits exactly, and neither split contains `[UNK]`.
- Training text averages about 2.2 characters per token and validation text slightly fewer. The merges were learned from training text, so they fit it a little better.
- The longest merges are frequent whole words (`▁father`, `▁blood`) and speaker names (`GLOUCESTER:`). Rare words split into short pieces; for example, `Citizen` becomes `▁C it i z en`.
- The unseen characters in `café ☃` become `[UNK]`, which is how the tokenizer handles symbols missing from the training text.


### 1.3 Token windows

Each split is encoded separately and cut into windows of `N + 1 = 65` tokens. For a window `z`, the input is `z[0:N]` and the target is `z[1:N+1]`, so the target at position `i` is the token that follows input position `i`. The same `N = 64` is used for every configuration.

- **Training windows** start at random offsets, so they may overlap and every position can appear as a target.
- **Validation windows** start at `0, N, 2N, …`; consecutive windows share one boundary token, so each target position is scored exactly once. The final short window is **dropped**, so the loss needs no padding mask.


In [5]:
CONTEXT_LEN = 64  # N

train_ids = torch.tensor(train_encoding.ids, dtype=torch.long)
val_ids = torch.tensor(val_encoding.ids, dtype=torch.long)


def get_train_batch(batch_size, generator):
    """Sample random, possibly overlapping training windows: inputs and targets of shape (B, N)."""
    starts = torch.randint(0, len(train_ids) - CONTEXT_LEN, (batch_size,), generator=generator)
    windows = torch.stack([train_ids[s:s + CONTEXT_LEN + 1] for s in starts.tolist()])
    return windows[:, :-1], windows[:, 1:]


num_val_windows = (len(val_ids) - 1) // CONTEXT_LEN
val_windows = torch.stack([val_ids[i * CONTEXT_LEN:i * CONTEXT_LEN + CONTEXT_LEN + 1] for i in range(num_val_windows)])
val_x, val_y = val_windows[:, :-1], val_windows[:, 1:]
num_val_targets = val_y.numel()

# Each validation target is the next unscored token: positions 1 .. num_val_targets, each once.
assert torch.equal(val_y.reshape(-1), val_ids[1:num_val_targets + 1])
assert torch.equal(val_x[:, 1:], val_y[:, :-1])

print(f"Train tokens                   {len(train_ids):,}")
print(f"Val tokens                     {len(val_ids):,}")
print(f"Validation windows             {num_val_windows:,} x {CONTEXT_LEN} targets")
print(f"Evaluated validation targets   {num_val_targets:,}")
print(f"Dropped (final short window)   {len(val_ids) - 1 - num_val_targets} targets")

x, y = get_train_batch(4, torch.Generator().manual_seed(SEED))
print(f"\nTraining batch shapes: inputs {tuple(x.shape)}, targets {tuple(y.shape)}")
print("First 8 positions of one training window:")
for position in range(8):
    print(f"  input {tokenizer.id_to_token(x[0, position].item())!r:>12}  ->  target {tokenizer.id_to_token(y[0, position].item())!r}")


Train tokens                   407,182
Val tokens                     105,963
Validation windows             1,655 x 64 targets
Evaluated validation targets   105,920
Dropped (final short window)   42 targets

Training batch shapes: inputs (4, 64), targets (4, 64)
First 8 positions of one training window:
  input         'ur'  ->  target 'n'
  input          'n'  ->  target ','
  input          ','  ->  target '▁good'
  input      '▁good'  ->  target '▁C'
  input         '▁C'  ->  target 'at'
  input         'at'  ->  target 'es'
  input         'es'  ->  target 'b'
  input          'b'  ->  target 'y,'


**Interpretation.**

- Training text yields 407,182 tokens. Validation yields 105,963 tokens, cut into 1,655 windows of 64 targets each.
- **105,920 validation targets** are scored. The final short window would have held 42 targets and is dropped; the first validation token is never a target because it has no preceding context.
- The assertions confirm that each validation target is scored once, and that each target row is its input row shifted one token to the left.
- The printed pairs show the shift directly: each target is the next input token.
